In [1]:
# colab_05_embedding_gorsel.py
# Store embedding 2B - "model magaza tiplerini kimse soylemeden buldu"
import json
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

DIZIN = "/content/drive/MyDrive/Colab Notebooks/datasets/rossman"
CIKTI = f"{DIZIN}/model"
meta = json.load(open(f"{DIZIN}/hazirlik/meta.json"))

E = np.load(f"{CIKTI}/magaza_embedding.npy")          # (1115, 24)
magazalar = np.array(meta["magazalar"])
store = pd.read_csv(f"{DIZIN}/store.csv").set_index("Store")
tip = store.loc[magazalar, "StoreType"].values
yelpaze = store.loc[magazalar, "Assortment"].values
tipik = np.expm1(np.array(meta["magaza_ort"]))
print("embedding:", E.shape)

pca = PCA(n_components=2).fit(E)
Z_pca = pca.transform(E)
Z_tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=42).fit_transform(E)
print(f"PCA aciklanan varyans: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
for t in np.unique(tip):
    s = tip == t
    ax[0].scatter(Z_pca[s,0], Z_pca[s,1], s=8, alpha=.6, label=f"tip {t} (n={s.sum()})")
    ax[1].scatter(Z_tsne[s,0], Z_tsne[s,1], s=8, alpha=.6, label=f"tip {t}")
ax[0].set_title(f"PCA ({pca.explained_variance_ratio_[:2].sum()*100:.0f}% varyans)")
ax[1].set_title("t-SNE"); ax[0].legend(fontsize=7); ax[1].legend(fontsize=7)
sc = ax[2].scatter(Z_tsne[:,0], Z_tsne[:,1], c=np.log1p(tipik), s=8, cmap="viridis")
ax[2].set_title("t-SNE - renk: log tipik ciro"); plt.colorbar(sc, ax=ax[2])
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{CIKTI}/embedding_2b.png", dpi=130)
print(f"grafik -> {CIKTI}/embedding_2b.png")

# ayrisma SAYISAL kanit: komsunun tipi tutuyor mu?
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=6).fit(E)
_, idx = nn.kneighbors(E)
for ad, etiket in [("magaza tipi", tip), ("urun yelpazesi", yelpaze)]:
    ayni = np.mean([np.mean(etiket[idx[i,1:]] == etiket[i]) for i in range(len(E))])
    taban = np.mean([(etiket == e).mean() for e in etiket])
    print(f"{ad}: en yakin 5 komsu ayni olma orani {ayni*100:.1f}% "
          f"(rastgele taban {taban*100:.1f}%)")

embedding: (1115, 24)
PCA aciklanan varyans: 52.4%
grafik -> /content/drive/MyDrive/Colab Notebooks/datasets/rossman/model/embedding_2b.png
magaza tipi: en yakin 5 komsu ayni olma orani 63.4% (rastgele taban 40.7%)
urun yelpazesi: en yakin 5 komsu ayni olma orani 60.6% (rastgele taban 49.5%)
